In [1]:
import torch
from mmasim_kernels.nv_ptx.rtx_blackwell import mma_kernels

torch.manual_seed(0)
HMMA_F32 = mma_kernels["m16n8k16.f32.f16.f16.f32"]
HMMA_F16 = mma_kernels["m16n8k16.f16.f16.f16.f16"]

In [ ]:
bsz = 100
A = torch.randn(bsz, 128, 128, device='cuda:0', dtype=torch.float16)
B = torch.randn(bsz, 128, 128, device='cuda:0', dtype=torch.float16)
A, B

(tensor([[[-9.2480e-01, -4.2529e-01, -2.6445e+00,  ..., -2.1277e-01,
           -3.3154e-01, -2.0227e-01],
          [-1.1455e+00, -5.7129e-01, -6.5088e-01,  ...,  1.3418e+00,
            3.3164e+00, -8.4668e-01],
          [ 2.3743e-01,  1.0889e+00, -2.4976e-01,  ...,  2.4304e-01,
            1.7029e-01, -2.1106e-01],
          ...,
          [ 4.5703e-01, -5.8545e-01, -6.3428e-01,  ..., -4.8633e-01,
           -7.6172e-01,  8.6035e-01],
          [-5.6592e-01,  8.8379e-01,  3.9246e-02,  ...,  3.5669e-01,
            6.9824e-01, -8.5754e-02],
          [ 5.4565e-02, -1.7810e-01, -3.3228e-01,  ..., -1.3159e-01,
           -4.2090e-01, -2.9551e+00]],
 
         [[ 1.3538e-01,  2.5342e-01,  9.0869e-01,  ...,  6.6260e-01,
           -1.1807e+00, -5.6104e-01],
          [-2.1399e-01, -2.1167e-01,  1.6670e+00,  ..., -2.3056e-02,
            7.5049e-01,  1.6055e+00],
          [-1.5605e+00, -3.2178e-01,  7.7246e-01,  ..., -3.6816e-01,
            1.2871e+00,  2.1472e-01],
          ...,
    

In [ ]:
D_HMMA_F16 = torch.zeros(bsz, 128, 128, device='cuda:0', dtype=torch.float16)
D_HMMA_F32 = torch.zeros(bsz, 128, 128, device='cuda:0', dtype=torch.float32)
D_real = A.double() @ B.double()
for t in range(bsz):
    for i in range(0, 128, 16):
        for j in range(0, 128, 8):
            for k in range(0, 128, 16):
                D_HMMA_F32[t, i:i+16, j:j+8] = HMMA_F32(A[t, i:i+16, k:k+16], B[t, k:k+16, j:j+8], D_HMMA_F32[t, i:i+16, j:j+8])
                D_HMMA_F16[t, i:i+16, j:j+8] = HMMA_F16(A[t, i:i+16, k:k+16], B[t, k:k+16, j:j+8], D_HMMA_F16[t, i:i+16, j:j+8])

In [ ]:
print("HMMA.F16 MSE:", (D_real - D_HMMA_F16).square().mean().item())
print("HMMA.F32 MSE:", (D_real - D_HMMA_F32).square().mean().item())
print("HMMA.F32 + Convert to F16 MSE:", (D_real - D_HMMA_F32.half()).square().mean().item())

HMMA.F16 MSE: 2.4739092130458212e-05
HMMA.F32 MSE: 4.46963683797174e-12
HMMA.F32 + Convert to F16 MSE: 5.479862290706972e-06
